In [4]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [5]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
# 1. 设置镜像（必须在导入 datasets 之前生效，或者确保环境已设置）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import time
import datasets
from datasets import load_dataset
from requests.exceptions import ChunkedEncodingError, ConnectionError, ReadTimeout


# 2. 配置下载参数
config = datasets.DownloadConfig(resume_download=True, max_retries=100)

# 3. 定义下载函数，包裹在循环中
def download_with_retry():
    max_retries_external = 50  # 外部强行重试次数
    attempt = 0
    
    while attempt < max_retries_external:
        try:
            print(f">>> 开始尝试下载 (第 {attempt + 1} 次)...")
            
            ds = load_dataset(
                "Shekswess/medical_llama3_instruct_dataset_short", 
                cache_dir="../../datasets/cache/medical-qa", 
                token=True, 
                download_config=config,
                # 某些情况下，多进程会导致断流更频繁，如果还报错，可以尝试把 num_proc 设为 1
                # num_proc=1 
            )
            
            print(">>> ✅ 下载并加载成功！")
            return ds
            
        except (ChunkedEncodingError, ConnectionError, ReadTimeout, Exception) as e:
            # 捕捉所有网络中断相关的错误
            print(f">>> ❌ 发生错误: {type(e).__name__}")
            print(f">>> 错误详情: {e}")
            print(f">>> ⚠️ 正在休眠 5 秒后准备断点续传...")
            time.sleep(5)
            attempt += 1
            
    raise RuntimeError("达到最大重试次数，下载失败。请检查网络或硬盘空间。")
    # 4. 执行下载
if __name__ == "__main__":
    ds = download_with_retry()
    print(ds)

>>> 开始尝试下载 (第 1 次)...
>>> ✅ 下载并加载成功！
DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction', 'prompt'],
        num_rows: 2000
    })
})


In [6]:
ds['train'][0]

{'output': 'Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: papillary, clear cell, small cell, and basaloid.',
 'input': "Can you provide an overview of the lung's squamous cell carcinoma?",
 'instruction': 'Answer the question truthfully, you are a medical professional.',
 'prompt': "<|start_header_id|>system<|end_header_id|> Answer the question truthfully, you are a medical professional.<|eot_id|><|start_header_id|>user<|end_header_id|> This is the question: Can you provide an overview of the lung's squamous cell carcinoma?<|eot_id|><|start_header_id|>assistant<|end_header_id|> Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: papillary, clear cell, small cell, and basaloid.<|eot_id|>"}

In [4]:
import random

# 假设 ds['train'] 是你的数据集
data = ds['train']

# 数据格式转换
formatted_data = []
for item in data:
    formatted_item = {
        "instruction": item["instruction"],
        "input": item["input"],
        "output": item["output"]
    }
    formatted_data.append(formatted_item)

# 随机打乱数据
random.shuffle(formatted_data)

# 数据划分
train_set = formatted_data[:500]
val_set = formatted_data[1200:1250]
test_set = formatted_data[1250:1260]

# 打印划分后的数据量
print(f"训练集大小: {len(train_set)}")
print(f"验证集大小: {len(val_set)}")
print(f"测试集大小: {len(test_set)}")

# 保存为 JSON 文件（可选）
import json

with open("../../datasets/processed/llamafactory/metadata/medical_train_set.json", "w") as f:
    json.dump(train_set, f, indent=4)

with open("../../datasets/processed/llamafactory/metadata/medical_val_set.json", "w") as f:
    json.dump(val_set, f, indent=4)

with open("../../datasets/processed/llamafactory/metadata/medical_test_set.json", "w") as f:
    json.dump(test_set, f, indent=4)

训练集大小: 500
验证集大小: 50
测试集大小: 10


In [9]:
from datasets import Dataset
import torch
from modelscope import snapshot_download,AutoTokenizer
from swanlab.integration.transformers import SwanLabCallback
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, TaskType, get_peft_model, PeftModel,get_peft_model_state_dict
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
)
import swanlab
import json

In [ ]:
def process_func_batch(examples):
    MAX_LENGTH = 2048
    input_ids, attention_mask, labels = [], [], []
    pixel_values, image_grid_thw = [], []
    
    # 遍历每个样本
    for instruction, input_text, output_text in zip(examples["instruction"], examples["input"], examples["output"]):
        # 构建 messages
        messages = [
                    {
                        "role": "system",
                        "content": [
                            {"type": "text", "text": instruction},
                        ],
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": input_text}
                        ]
                    }
                ]
        
        # 处理视觉信息
        image_inputs, video_inputs = process_vision_info(messages)
        
        # 如果 image_inputs 或 video_inputs 为空，则设为 None
        if not image_inputs:
            image_inputs = None
        if not video_inputs:
            video_inputs = None
        
        # 使用 processor 处理文本
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        # 处理文本输入
        inputs = processor(
            text=[text],
            images=image_inputs,  # 添加图像输入
            videos=video_inputs,  # 添加视频输入
            padding=False,  # 先不填充
            return_tensors="pt",
        )

        # 提取 input_ids 和 attention_mask
        inputs_dict = {key: value.tolist() for key, value in inputs.items()}
        instruction_input_ids = inputs_dict['input_ids'][0]
        instruction_attention_mask = inputs_dict['attention_mask'][0]

        # 处理输出文本
        response = tokenizer(f"{output_text}", add_special_tokens=False)
        response_input_ids = response['input_ids']
        response_attention_mask = response['attention_mask']

        # 计算剩余可用长度给 response
        remaining_length = MAX_LENGTH - len(instruction_input_ids) - 1  # 减去一个 PAD token 的空间

        if remaining_length < 0:
            # 如果指令部分已经超过最大长度，则需要截断指令部分
            truncation_length = len(instruction_input_ids) + remaining_length
            instruction_input_ids = instruction_input_ids[:truncation_length]
            instruction_attention_mask = instruction_attention_mask[:truncation_length]
            remaining_length = 0

        # 截断 response 部分以适应剩余空间
        current_input_ids = (
            instruction_input_ids + response_input_ids[:remaining_length] + [tokenizer.pad_token_id]
        )

        current_attention_mask = (
            instruction_attention_mask + response_attention_mask[:remaining_length] + [1]
        )
        current_labels = (
            [-100] * len(instruction_input_ids) +
            response_input_ids[:remaining_length] +
            [tokenizer.pad_token_id]
        )
        
        # 填充到 MAX_LENGTH
        if len(current_input_ids) < MAX_LENGTH:
            current_input_ids += [tokenizer.pad_token_id] * (MAX_LENGTH - len(current_input_ids))
            current_attention_mask += [0] * (MAX_LENGTH - len(current_attention_mask))
            current_labels += [-100] * (MAX_LENGTH - len(current_labels))

        # 添加到列表中
        input_ids.append(current_input_ids)
        attention_mask.append(current_attention_mask)
        labels.append(current_labels)
        
        # 处理 pixel_values 和 image_grid_thw
        if image_inputs is not None:
            pixel_values.append(inputs_dict['pixel_values'][0])
            image_grid_thw.append(torch.tensor(inputs_dict['image_grid_thw'][0]).squeeze(0))
        else:
            pixel_values.append(None)
            image_grid_thw.append(None)

    # 返回结果
    return {
        "input_ids": torch.tensor(input_ids),  # 转换为 torch.Tensor
        "attention_mask": torch.tensor(attention_mask),  # 转换为 torch.Tensor
        "labels": torch.tensor(labels),  # 转换为 torch.Tensor
        "pixel_values": torch.tensor(pixel_values) if any(pixel_values) else pixel_values,  # 转换为 torch.Tensor 或 None
        "image_grid_thw": torch.stack(image_grid_thw) if any(image_grid_thw) else image_grid_thw  # 转换为 torch.Tensor 或 None
    }

# 加载数据集
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("/root/autodl-tmp/Qwen/Qwen2.5-VL-7B-Instruct/", use_fast=True)
min_pixels = 256 * 28 * 28
max_pixels = 1280 * 28 * 28
processor = AutoProcessor.from_pretrained("/root/autodl-tmp/Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels, use_fast=True)
train_ds = Dataset.from_json("../../datasets/processed/llamafactory/metadata/medical_train_set.json")
train_dataset = train_ds.map(process_func_batch, batched=True)